# Lab 03 extra: How ReAct text differs from native tool calls

**Going further, for Lab 03**

## Goal

Before chat APIs had tool calls, the agent loop was written in plain text. The model wrote a thought and an action
as ordinary lines, our code read the action out of the text and ran it, and the result went back as one more line
of text. The best-known form of this is **ReAct**, short for reason and act (Yao and others, 2022). We build it
once on the envelope hunt, beside the same hunt with native tool calls, to see which jobs the API's tool calls took
over from our prompt and our parser.

This notebook is for understanding only. Agents today ask for tools with native tool calls, as in the two main
notebooks of this lab. The name has stayed: frameworks still say "ReAct agent" for the loop of reasoning, acting and
reading the result, and they run it on native tool calls.

## What you will learn

- How a text format asks for a tool: the format lives in the system prompt, and the reply is ordinary text.
- How to read the action out of a reply, run it, and send the result back as an observation.
- How to write the ReAct loop and read its transcript.
- Which jobs native tool calls do for us that the text loop leaves to the prompt and the parser.
- What the text format does not enforce, and why.

## Setup

The same first cell as every lab: it reads the model name from `models.env` and takes the key from `.env` without
printing it.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
from openai import OpenAI, omit

client = OpenAI(timeout=60)

The client has a 60-second limit on each request. Every call passes `reasoning_effort=EFFORT or omit`, where `omit`
leaves the argument out when `models.env` sets no effort.

---

## The hunt and its one tool

The game is an envelope hunt. A prize is in one of five numbered envelopes, and the others hold a clue or nothing.
The envelopes are a dictionary, and `open_envelope` is a plain Python function that prints a `TOOL CALLED` line and
returns what the envelope holds. This notebook restates them, so it runs on its own.

In [3]:
ENVELOPES = {
    1: "Clue: look in envelope 4.",
    2: "The prize! Two tickets to the cinema.",
    3: "This envelope is empty.",
    4: "Clue: look in envelope 2.",
    5: "This envelope is empty.",
}


def open_envelope(number: int) -> str:
    """Open one envelope and read what is inside."""
    print(f"TOOL CALLED: open_envelope({number})")
    return ENVELOPES.get(number, "There is no envelope with that number.")


QUESTION = "Where is the prize? Start with envelope 1."

The right path, for checking every run below: envelope 1 points to 4, 4 points to 2, and 2 holds the prize.

---

## Ask for a tool in plain text

There is no `tools=` argument in this part. Everything the model needs is in the system prompt:

- the action it may ask for, and what the action returns;
- the **format** of a reply that asks for it: three lines, `Thought:`, `Action:` and `Action Input:`;
- the format of a reply that ends the hunt: `Thought:` and `Final Answer:`;
- a rule: never write an `Observation:` line. That line is ours. It is how the result of the action will come back.

In [4]:
REACT_SYSTEM = """You are playing an envelope hunt. A prize is hidden in one of the numbered envelopes.
You cannot open an envelope yourself. You have one action:

open_envelope   input: the envelope's number   returns: what is inside

Reply with exactly these three lines:
Thought: <what you need next, and why>
Action: open_envelope
Action Input: <a number>

When you know where the prize is, reply with these two lines instead:
Thought: <why you are done>
Final Answer: <the answer in one sentence>

Never write an Observation line yourself. Our code opens the envelope and sends you the Observation."""

messages = [
    {"role": "system", "content": REACT_SYSTEM},
    {"role": "user", "content": QUESTION},
]

response = client.chat.completions.create(model=MODEL, messages=messages, reasoning_effort=EFFORT or omit)
text = response.choices[0].message.content

print("finish_reason:", response.choices[0].finish_reason)
print(text)

finish_reason: stop
Thought: I need to check envelope 1 to find out whether it contains the prize.
Action: open_envelope
Action Input: 1


In our run the reply has the three lines we asked for: a thought (it needs to check envelope 1), the action
`open_envelope`, and the input `1`.

**Notice** the `finish_reason`: `stop`. As far as the API can tell, the model wrote ordinary text and finished. The
request for a tool is inside that text, and only our code can find it. No `TOOL CALLED` line was printed: nothing
has run.

---

## Read the action out of the text

Our code has to find the action in the reply. Each line that matters starts with a label and a colon, so we split
on the first colon. One line first:

In [5]:
print("Action Input: 1".split(":", 1))

['Action Input', ' 1']


`split(":", 1)` cuts the string at the first colon only and returns a list of two parts: the label, and everything
after it. `[1]` takes the second part, and `.strip()` removes the space in front of it.

`parse_react` does that for every line of a reply. It keeps the action's name, the action's input and the final
answer, and returns `None` for any of them that the reply does not hold.

In [6]:
def parse_react(text):
    action = action_input = final = None

    for line in text.splitlines():
        if line.startswith("Action:"):
            action = line.split(":", 1)[1].strip()
        elif line.startswith("Action Input:"):
            action_input = line.split(":", 1)[1].strip()
        elif line.startswith("Final Answer:"):
            final = line.split(":", 1)[1].strip()

    return action, action_input, final


print(parse_react(text))

('open_envelope', '1', None)


Three values: the action `open_envelope`, the input `'1'`, and `None` for the final answer, because this reply has
none.

The input is a string. Our function takes a whole number, so we turn `"1"` into `1` with `int(...)`, run the
function, and send the result back.

**Why does the result go back as a `user` message?** Chat Completions has a `tool` role, and a `tool` message must
answer a tool call that the API itself recorded. Our text format makes no such call: to the API the `Action:` line
is ordinary assistant text. So the result goes back as a user message, and the label `Observation:` tells the model
it is a result and not the person speaking.

In [7]:
action, action_input, final = parse_react(text)

observation = open_envelope(int(action_input))
print("Observation:", observation)

messages.append({"role": "assistant", "content": text})
messages.append({"role": "user", "content": "Observation: " + observation})

TOOL CALLED: open_envelope(1)
Observation: Clue: look in envelope 4.


Now `TOOL CALLED` is printed, by our own line, and the clue from envelope 1 is in the list as an observation. That
was one round by hand.

---

## The ReAct loop

The loop has the same shape as the loop with native tool calls. Only the places where it reads the reply are
different:

1. Send the whole `messages` list. There is no `tools=`.
2. Add the reply's text to the list as an `assistant` message.
3. Parse the text. If it holds a `Final Answer:`, stop.
4. Otherwise run the action and add the result as a `user` message that starts with `Observation:`.
5. Go round again, up to a turn cap.

`range(1, 9)` is the turn cap here: eight turns at most. Each pass prints the turn's tokens (what the call sent, and
what the model wrote) and the reply's text. We start from a fresh list.

In [8]:
messages = [
    {"role": "system", "content": REACT_SYSTEM},
    {"role": "user", "content": QUESTION},
]

for turn in range(1, 9):
    response = client.chat.completions.create(model=MODEL, messages=messages, reasoning_effort=EFFORT or omit)
    text = response.choices[0].message.content
    messages.append({"role": "assistant", "content": text})

    print(f"--- turn {turn} | prompt tokens {response.usage.prompt_tokens} | output tokens {response.usage.completion_tokens}")
    print(text)

    action, action_input, final = parse_react(text)
    if final is not None:
        break

    observation = open_envelope(int(action_input))
    print("Observation:", observation)
    messages.append({"role": "user", "content": "Observation: " + observation})

print()
print("answer:", final)

--- turn 1 | prompt tokens 150 | output tokens 43
Thought: I need to inspect envelope 1 to find out whether it contains the prize or points to another envelope.
Action: open_envelope
Action Input: 1
TOOL CALLED: open_envelope(1)
Observation: Clue: look in envelope 4.


--- turn 2 | prompt tokens 205 | output tokens 35
Thought: Envelope 1 points to envelope 4, so I need to inspect envelope 4 next.
Action: open_envelope
Action Input: 4
TOOL CALLED: open_envelope(4)
Observation: Clue: look in envelope 2.


--- turn 3 | prompt tokens 258 | output tokens 35
Thought: Envelope 4 points to envelope 2, so I need to inspect envelope 2 next.
Action: open_envelope
Action Input: 2
TOOL CALLED: open_envelope(2)
Observation: The prize! Two tickets to the cinema.


--- turn 4 | prompt tokens 311 | output tokens 34
Thought: Envelope 2 contained the prize, so I’m done.
Final Answer: The prize is in envelope 2: two tickets to the cinema.

answer: The prize is in envelope 2: two tickets to the cinema.


Read the transcript like the notes of a person working through the hunt. Each `Thought:` says what the model needs
next, each `Observation:` is our code's answer, and the next thought uses it: *envelope 1 points to envelope 4*. The
run took four turns: three actions, for envelopes 1, 4 and 2, and a final answer that names envelope 2, the right one.

As with the first reply, the API saw four pieces of ordinary text. Our parser alone decided which of them ended the
run.

---

## The same hunt with native tool calls

Now the version the main notebooks use. The tool is described in a schema and sent in `tools=`. The model's request
comes back in `tool_calls`, a field the API fills in, and the result goes back as a `tool` message.

In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "open_envelope",
            "description": "Open one envelope and read what is inside.",
            "parameters": {
                "type": "object",
                "properties": {"number": {"type": "integer", "description": "The envelope's number"}},
                "required": ["number"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

NATIVE_SYSTEM = (
    "You are playing an envelope hunt. A prize is hidden in one of the numbered envelopes. "
    "Open envelopes with open_envelope and follow the clues. Answer in one sentence."
)

The loop is the one from notebook 1: send the list with `tools=`, add the reply, stop unless `finish_reason` is
`tool_calls`, run each call and add a `tool` message for it. The same `range(1, 9)` caps it at eight turns. The loop prints the same token counts as the ReAct loop, so we can put the two runs side by side.

In [10]:
messages = [
    {"role": "system", "content": NATIVE_SYSTEM},
    {"role": "user", "content": QUESTION},
]

for turn in range(1, 9):
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, reasoning_effort=EFFORT or omit
    )
    reply = response.choices[0].message
    messages.append(reply)

    print(f"--- turn {turn} | prompt tokens {response.usage.prompt_tokens} | output tokens {response.usage.completion_tokens}"
          f" | finish_reason {response.choices[0].finish_reason}")

    if response.choices[0].finish_reason != "tool_calls":
        break

    for call in reply.tool_calls:
        result = open_envelope(**json.loads(call.function.arguments))
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

print()
print("answer:", reply.content)

--- turn 1 | prompt tokens 182 | output tokens 18 | finish_reason tool_calls
TOOL CALLED: open_envelope(1)


--- turn 2 | prompt tokens 221 | output tokens 18 | finish_reason tool_calls
TOOL CALLED: open_envelope(4)


--- turn 3 | prompt tokens 260 | output tokens 18 | finish_reason tool_calls
TOOL CALLED: open_envelope(2)


--- turn 4 | prompt tokens 299 | output tokens 17 | finish_reason stop

answer: The prize is in envelope 2: two tickets to the cinema.


The same three envelopes in the same order, the same answer, and four turns again. Two differences show in the
printed lines:

- **The API says when a tool is wanted.** Turns 1 to 3 end with `finish_reason` `tool_calls`, and turn 4 with
  `stop`. Our code read no text to find that out.
- **The output tokens.** A native request for a tool took 18 output tokens a turn here. A ReAct turn took 34 to 43,
  because the thought and the labels are output too. Over the whole run that is 71 output tokens against 147.

### Who does which job

| Job | ReAct text | Native tool calls |
|---|---|---|
| Tell the model which tools exist | our system prompt, in words | the schema in `tools=` |
| Say that a tool is wanted | a line in the reply's text | `finish_reason` is `tool_calls` |
| Find the tool and its arguments | our parser, `parse_react` | the API: `call.function.name` and `call.function.arguments` |
| Make the arguments valid | nothing: `int("two")` would raise | the API, with `strict: True` |
| Send the result back | a `user` message that starts with `Observation:` | a `tool` message with the call's id |
| Know the run is over | our parser finds `Final Answer:` | `finish_reason` is not `tool_calls` |
| Several tools in one reply | not in this format | a list of `tool_calls` |

Every row of the middle column is something we wrote and have to keep working. Every row of the right column is
done by the API.

What ReAct gives in return is the `Thought:` line: a readable note before each action. It is text the model wrote
about its next step, not a view of how the reply was computed, and it is paid for as output tokens on every turn.

---

## What the text format does not enforce

The format is only a request in the prompt. Two things show what that means.

### Nothing stops the model at the `Observation:` line

The risk in a text loop is a reply that keeps writing past its `Action Input:` line: an `Observation:` the model
made up, and then a `Final Answer:` built on it. The classic guard is a **stop sequence**: `stop=["Observation:"]`
tells the API to cut the reply off as soon as that text appears. We try it.

In [11]:
try:
    client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Say hello."}],
        stop=["Observation:"],
        reasoning_effort=EFFORT or omit,
    )
    print("the request was accepted")
except Exception as error:
    print(type(error).__name__, "|", error)

BadRequestError | Error code: 400 - Unsupported parameter: 'stop' is not supported with this model.


The API refused the request with a 400: `stop` is not supported with this model. Some models accept a stop
sequence and some do not. Without one, the only thing that keeps the model from writing its own observation is the
sentence in our system prompt.

### A reply that writes its own observation passes the parser

To see what the rule in the prompt is protecting us from, we type a reply by hand. This text was **not** written by
the model. It is what a reply would look like if the model ignored the rule and went on writing after its action.

In [12]:
made_up_reply = """Thought: I need to open envelope 1.
Action: open_envelope
Action Input: 1
Observation: The prize! A new bicycle.
Thought: Envelope 1 holds the prize, so I am done.
Final Answer: The prize is in envelope 1: a new bicycle."""

print(parse_react(made_up_reply))

('open_envelope', '1', 'The prize is in envelope 1: a new bicycle.')


`parse_react` found a final answer, so our loop would stop here and return it: a bicycle in envelope 1. No envelope
was opened. There is no `TOOL CALLED` line, because no code of ours ran. The parser reads lines. It cannot tell an
observation our code wrote from one that came inside the reply.

With native tool calls this cannot happen in the same way. A `tool` message can only come from our code, and the
API refuses a `tool` message that answers no recorded call.

In the runs we made on this model the rule in the prompt held: no reply wrote its own `Observation:`. A rule that held
in the runs we saw is still only a sentence in a prompt. The text loop did go wrong in another way. In 2 of the 10
runs we made, the model read the first clue, *look in envelope 4*, as the answer, and stopped after one envelope with
a final answer that names envelope 4.

**In production:** use native tool calls. Keep ReAct as a way to understand what the loop does, and for the rare
model that has no tool calling. When a readable record of each step is needed, it comes from tracing every call of
the run, not from a thought line in the prompt.

---

## What you can do now

- Ask for a tool in a text format, and say why the reply's `finish_reason` is `stop`.
- Parse an action out of a reply, run it, and send the result back as an observation.
- Write the ReAct loop with a turn cap and read its transcript.
- Name the jobs that native tool calls do for us and the text loop leaves to our prompt and parser.
- Explain what a made-up observation does to a text loop, and why a `tool` message cannot be made up by the model.

## Resources

- [Yao and others, the ReAct paper (2022)](https://arxiv.org/abs/2210.03629): the paper that named the pattern,
  reasoning and acting written as one text
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): native tool calls,
  schemas and strict mode
- [OpenAI: Chat Completions API reference](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create):
  `finish_reason`, `stop` and the message roles

Back to [notebook 1](03a_meet_the_agent_loop.ipynb) and [notebook 2](03b_agent_loop_at_the_desk.ipynb).